In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np

from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

from captum.attr import LayerIntegratedGradients

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [14]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim=512, num_heads=8):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.q = nn.Linear(embed_dim, embed_dim)
        self.k = nn.Linear(embed_dim, embed_dim)
        self.v = nn.Linear(embed_dim, embed_dim)
        self.fc = nn.Linear(embed_dim, embed_dim)

    def forward(self, x, mask=None):
        B, T, C = x.shape

        q = self.q(x)
        k = self.k(x)
        v = self.v(x)

        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1,2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1,2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1,2)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if mask is not None:
            mask = mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(mask == 0, -1e9)

        attn = torch.softmax(scores, dim=-1)
        out = attn @ v

        out = out.transpose(1,2).contiguous().view(B, T, C)
        out = self.out(out)

        return out

In [15]:
class FeedForward(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, 4*embed_dim),
            nn.ReLU(),
            nn.Linear(4*embed_dim, embed_dim)
        )

    def forward(self, x):
        return self.net(x)

In [16]:
class EncoderLayer(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()
        self.attn = MultiHeadSelfAttention(embed_dim)
        self.ff = FeedForward(embed_dim)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.norm1(x + self.attn(x))
        x = self.norm2(x + self.ff(x))
        return x

In [17]:
import torch
import torch.nn as nn
import math

class SimpleBERT(nn.Module):
    def __init__(self, vocab_size, embed_dim=512, max_len=128, num_layers=2):
        super().__init__()

        self.token_embeddings = nn.Embedding(vocab_size, embed_dim)
        self.position_embeddings = nn.Embedding(max_len, embed_dim)
        self.token_type_embeddings = nn.Embedding(2, embed_dim)


        self.layers = nn.ModuleList([
            EncoderLayer(embed_dim) for _ in range(num_layers)
        ])


        self.pooler = nn.Linear(embed_dim, embed_dim)
        self.classifier = nn.Linear(embed_dim, 2)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        B, T = input_ids.shape


        pos = torch.arange(0, T).unsqueeze(0).repeat(B,1).to(input_ids.device)


        if token_type_ids is None:
            token_type_ids = torch.zeros_like(input_ids)


        token_emb = self.token_embeddings(input_ids)
        pos_emb = self.position_embeddings(pos)
        type_emb = self.token_type_embeddings(token_type_ids)

        x = token_emb + pos_emb + type_emb


        print("Token Emb:", token_emb.shape)
        print("Pos Emb:", pos_emb.shape)


        for layer in self.layers:
            x = layer(x, attention_mask)

        print("Encoder Output:", x.shape)


        cls_token = x[:, 0]
        pooled = torch.tanh(self.pooler(cls_token))
        out = self.classifier(pooled)

        print("Output:", out.shape)

        return out

In [18]:
class EncoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads=8, ff_dim=2048):
        super().__init__()

        self.attention = MultiHeadSelfAttention(embed_dim, num_heads)
        self.norm1 = nn.LayerNorm(embed_dim)

        self.ff = FeedForward(embed_dim, ff_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x, mask=None):
        attn_out = self.attention(x, mask)
        x = self.norm1(x + attn_out)
        ff_out = self.ff(x)
        x = self.norm2(x + ff_out)

        return x

In [21]:
train = pd.read_csv("/all_test_public.tsv", sep='\t')
test = pd.read_csv("/all_test_public.tsv", sep='\t')
val = pd.read_csv("/all_validate.tsv", sep='\t')

df = pd.concat([train, test, val])
df = df[['clean_title', '2_way_label', 'id']]
df.dropna(inplace=True)

print("Original size:", len(df))

Original size: 253498


In [22]:
fake = df[df['2_way_label'] == 1].sample(2500, random_state=42)
real = df[df['2_way_label'] == 0].sample(2500, random_state=42)

balanced = pd.concat([fake, real])

print(balanced['2_way_label'].value_counts())

2_way_label
1    2500
0    2500
Name: count, dtype: int64


In [23]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [24]:
train_df, test_df = train_test_split(balanced, test_size=0.2, stratify=balanced['2_way_label'])

print("Train:", len(train_df))
print("Test:", len(test_df))

Train: 4000
Test: 1000


In [25]:
def tokenize(data):
    return tokenizer(
        list(data['clean_title']),
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

train_enc = tokenize(train_df)
test_enc = tokenize(test_df)

In [26]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
model.to(device)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [27]:
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

train_labels = torch.tensor(train_df['2_way_label'].values).to(device)

In [28]:
import torch
torch.cuda.empty_cache()

In [29]:
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': torch.tensor(self.labels[idx])
        }

    def __len__(self):
        return len(self.labels)

In [30]:
train_dataset = CustomDataset(train_enc, train_df['2_way_label'].values)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

In [31]:
EPOCHS = 2
BATCH_SIZE = 8
LR = 2e-5
MAX_LEN = 128

print("Hyperparameters:")
print(f"Epochs: {EPOCHS}, Batch Size: {BATCH_SIZE}, LR: {LR}, Max Length: {MAX_LEN}")
model.train()

for epoch in range(EPOCHS):
    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}/{EPOCHS} done, Loss: {loss.item():.4f}")

Hyperparameters:
Epochs: 2, Batch Size: 8, LR: 2e-05, Max Length: 128
Epoch 1/2 done, Loss: 0.2502
Epoch 2/2 done, Loss: 0.1455


In [32]:
from torch.utils.data import DataLoader

test_dataset = CustomDataset(test_enc, test_df['2_way_label'].values)

test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Metrics
acc = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')

print("Accuracy:", acc)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

print("Confusion Matrix:\n", confusion_matrix(all_labels, all_preds))
from sklearn.metrics import classification_report

print("Classification Report:\n")
print(classification_report(all_labels, all_preds, digits=4))

Accuracy: 0.796
Precision: 0.7587412587412588
Recall: 0.868
F1: 0.8097014925373134
Confusion Matrix:
 [[362 138]
 [ 66 434]]
Classification Report:

              precision    recall  f1-score   support

           0     0.8458    0.7240    0.7802       500
           1     0.7587    0.8680    0.8097       500

    accuracy                         0.7960      1000
   macro avg     0.8023    0.7960    0.7949      1000
weighted avg     0.8023    0.7960    0.7949      1000



In [33]:
def forward_func(inputs_embeds, attention_mask):
    outputs = model(
        inputs_embeds=inputs_embeds,
        attention_mask=attention_mask
    )
    return outputs.logits

In [34]:
lig = LayerIntegratedGradients(forward_func, model.bert.embeddings)

In [35]:
sample = test_df.iloc[0]['clean_title']

enc = tokenizer(sample, return_tensors="pt", truncation=True, padding=True, max_length=128)

input_ids = enc['input_ids'].to(device)
attention_mask = enc['attention_mask'].to(device)

# Convert to embeddings
embeddings = model.bert.embeddings.word_embeddings(input_ids)

# Baseline (all zeros)
baseline = torch.zeros_like(embeddings).to(device)

attributions, delta = lig.attribute(
    inputs=embeddings,
    baselines=baseline,
    additional_forward_args=(attention_mask,),
    target=1,
    n_steps=50,   # increase if delta is high
    return_convergence_delta=True
)

print("Convergence Delta:", delta)

Convergence Delta: tensor([-0.8221], device='cuda:0')


In [40]:
model.eval()

from captum.attr import LayerIntegratedGradients

lig = LayerIntegratedGradients(forward_func, model.bert.embeddings)

# Select 3 fake and 3 real samples
fake_samples = test_df[test_df['2_way_label'] == 1].head(3)
real_samples = test_df[test_df['2_way_label'] == 0].head(3)

samples = pd.concat([fake_samples, real_samples])

for idx, row in samples.iterrows():
    text = row['clean_title']
    label = row['2_way_label']

    encoding = tokenizer(
        text,
        return_tensors='pt',
        padding='max_length',
        truncation=True,
        max_length=128
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    embeddings = model.bert.embeddings(input_ids)

    baseline = torch.zeros_like(embeddings)

    attributions, delta = lig.attribute(
        inputs=embeddings,
        baselines=baseline,
        additional_forward_args=(attention_mask,),
        target=label,
        n_steps=100,   # increased for better convergence
        return_convergence_delta=True
    )

    # Aggregate
    attr = attributions.sum(dim=-1).squeeze(0)
    attr = attr / torch.norm(attr)

    tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze(0))

    # Remove special tokens
    filtered = [(tok, score.item()) for tok, score in zip(tokens, attr)
                if tok not in ['[CLS]', '[SEP]', '[PAD]']]

    # Top 5 tokens
    top_tokens = sorted(filtered, key=lambda x: abs(x[1]), reverse=True)[:5]

    print("\n==============================")
    print("Text:", text)
    pred = torch.argmax(model(input_ids, attention_mask=attention_mask).logits, dim=1).item()
    print("True Label:", label, "Predicted Label:", pred)
    print("Top 5 Important Tokens:")
    for tok, score in top_tokens:
        print(f"{tok}: {score:.4f}")

    print("Convergence Delta:", delta.item())


Text: santa uses sign language with deaf boy youtube
True Label: 1 Predicted Label: 1
Top 5 Important Tokens:
boy: 0.2753
uses: 0.2184
deaf: 0.1797
sign: 0.1315
language: 0.1290
Convergence Delta: -0.08443737030029297

Text: honeywell workers say lockout aims to destroy union its corporate greed
True Label: 1 Predicted Label: 1
Top 5 Important Tokens:
workers: 0.5497
destroy: -0.3426
say: 0.3111
aims: 0.2211
corporate: 0.2187
Convergence Delta: 0.2750976085662842

Text: this dog on a raft
True Label: 1 Predicted Label: 1
Top 5 Important Tokens:
dog: 0.7456
this: 0.2946
raft: 0.0696
on: 0.0371
a: -0.0018
Convergence Delta: -1.30391263961792

Text: a koala hiding in a tree
True Label: 0 Predicted Label: 1
Top 5 Important Tokens:
##ala: -0.5405
in: 0.4366
a: 0.4047
a: -0.3486
hiding: 0.2110
Convergence Delta: 1.1702884435653687

Text: a woodpecker put holes in my wood fence
True Label: 0 Predicted Label: 1
Top 5 Important Tokens:
##pe: -0.1661
in: 0.1096
wood: -0.1077
fence: -0.1019
hole